In [9]:
from helpers import json_to_llm_string
import random
import pandas as pd

from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from transformers import AutoTokenizer

In [14]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
llm = LLM(
    model=MODEL_NAME,
    max_model_len=8192,
    max_num_batched_tokens=8192,
    tokenizer_mode="auto",
)

INFO 03-02 14:10:05 [utils.py:233] non-default args: {'max_model_len': 8192, 'max_num_batched_tokens': 8192, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
INFO 03-02 14:10:06 [model.py:547] Resolved architecture: Qwen2ForCausalLM
INFO 03-02 14:10:06 [model.py:1510] Using max model len 8192
INFO 03-02 14:10:06 [arg_utils.py:1166] Chunked prefill is not supported for ARM and POWER and S390X CPUs; disabling it for V1 backend.


AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended

In [15]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

llm = LLM(
    model=MODEL,
    max_model_len=2048,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

params = SamplingParams(temperature=0.0, max_tokens=20)

prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a classifier. Output JSON only."},
        {"role": "user", "content": "Drug A increases toxicity of Drug B."}
    ],
    tokenize=False,
    add_generation_prompt=True
)

out = llm.generate([prompt], params)
print(out[0].outputs[0].text)

INFO 03-02 14:10:27 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-02 14:10:27 [model.py:547] Resolved architecture: Qwen2ForCausalLM
INFO 03-02 14:10:27 [model.py:1510] Using max model len 2048
INFO 03-02 14:10:27 [arg_utils.py:1166] Chunked prefill is not supported for ARM and POWER and S390X CPUs; disabling it for V1 backend.


AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-0.5B-Instruct",
    # torch_dtype="auto", # 
    # device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

prompt = "Give me a short introduction to large language model."


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
SYSTEM_PROMPT = """You are an expert in pharmaceutical drugs.
Your role is to determine how two classify pairs of drugs into
an interaction type, or state that there is "No interaction".
You do not need to provide any reasoning, but your decision
should be grounded in scientific facts and drug information
that is presented to you.
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt")

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


'A large language model is a type of artificial intelligence model that can generate text from large amounts of text data, such as Wikipedia articles or news articles. These models can be trained on large datasets and can process vast amounts of information in a matter of seconds or minutes. They have been used for tasks such as answering questions, summarizing text, generating text based on patterns found in the text, and even creating visual content such as images and videos. Large language models are becoming increasingly popular due to their ability to produce high-quality output quickly and accurately, but they also raise concerns about privacy and ethical implications.'